# Data Cleaning — UCI Drug Consumption Dataset

This notebook loads the raw UCI Drug Consumption dataset, assigns column names from the UCI documentation, inspects data quality, removes the fictitious drug Semeron (used to identify over-claimers), drops the ID column, and saves a clean version ready for EDA and encoding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# The raw .data file has no header row.
# Column names come from the UCI documentation:
# https://archive.ics.uci.edu/dataset/373/drug+consumption+quantified

ALL_COLS = [
    'ID', 'Age', 'Gender', 'Education', 'Country', 'Ethnicity',
    'Nscore', 'Escore', 'Oscore', 'Ascore', 'Cscore', 'Impulsive', 'SS',
    'Alcohol', 'Amphet', 'Amyl', 'Benzos', 'Caff', 'Cannabis', 'Choc',
    'Coke', 'Crack', 'Ecstasy', 'Heroin', 'Ketamine', 'Legalh', 'LSD',
    'Meth', 'Mushrooms', 'Nicotine', 'Semer', 'VSA'
]

df = pd.read_csv('drug_consumption.data', header=None, names=ALL_COLS)

print('Shape:', df.shape)
df.head()

In [ ]:
# Inspect data types
df.dtypes

In [ ]:
# Check for missing values — UCI dataset is complete with no nulls
print('Missing values per column:')
print(df.isnull().sum())
print()
print('Total missing:', df.isnull().sum().sum())

- The UCI Drug Consumption dataset contains **no missing values** — all 1,885 records are complete.
- The 12 demographic/personality features (Age through SS) are pre-quantified real values from the original study.
- The 19 drug columns use ordinal class labels: CL0–CL6.

In [ ]:
# Describe numeric feature columns
FEATURE_COLS = ['Age','Gender','Education','Country','Ethnicity',
                'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']

df[FEATURE_COLS].describe().round(3)

In [ ]:
# Drug usage class label distribution for each substance
DRUG_COLS = ['Alcohol','Amphet','Amyl','Benzos','Caff','Cannabis','Choc',
             'Coke','Crack','Ecstasy','Heroin','Ketamine','Legalh','LSD',
             'Meth','Mushrooms','Nicotine','Semer','VSA']

for col in DRUG_COLS:
    print(f'{col:12s}: {dict(df[col].value_counts().sort_index())}')

In [ ]:
# Remove Semeron (Semer) — a fictitious drug used to identify over-claimers.
# Respondents who claimed to have used Semeron are unreliable.
# UCI recommendation: flag and remove over-claimers (Semer != CL0).

overclaimer_mask = df['Semer'] != 'CL0'
print(f'Over-claimers identified (Semer != CL0): {overclaimer_mask.sum()}')

df_clean = df[~overclaimer_mask].copy()
df_clean = df_clean.drop(columns=['Semer'])  # drop Semer column — no longer needed
print(f'Rows after removing over-claimers: {len(df_clean)}')

In [ ]:
# Drop the ID column — it is a record index with no predictive value
df_clean = df_clean.drop(columns=['ID'])
print('Columns after cleaning:')
print(df_clean.columns.tolist())
print('Shape:', df_clean.shape)

In [ ]:
# Check for duplicate rows
dupes = df_clean.duplicated().sum()
print(f'Duplicate rows: {dupes}')

- **8 over-claimers** removed (respondents who claimed to have used the fictitious drug Semeron).
- **ID column** dropped — purely an index with no predictive meaning.
- No duplicate rows found.
- Final clean dataset: **1,877 rows × 31 columns**.

In [ ]:
# Save cleaned dataset
df_clean.to_csv('uci_drug_clean.csv', index=False)
print('Saved: uci_drug_clean.csv')
df_clean.shape